## Logistic Regression, From Scratch

See also: [05-xgboost-from-scratch.ipynb](05-xgboost-from-scratch.ipynb) (same p-y gradient extended to boosted trees).

Plain terms: start with a random guess for the weights. For every point, check how wrong that guess is, then nudge the weights a small step in the direction that makes the guess less wrong. Repeat until the wrongness stops shrinking. Sigmoid turns the raw guess into a probability; gradient descent is just this repeated small-correction process.

1. Probability, odds, log-odds: $p=\text{successes}/\text{total}\in[0,1]$. Odds $=p/(1-p)\in[0,\infty)$. Log-odds $=\ln(\text{odds})\in(-\infty,\infty)$.

| state | p | odds | log-odds |
|---|---|---|---|
| impossible | 0.0 | 0 | $-\infty$ |
| unlikely | 0.2 | 0.25 | -1.386 |
| coin toss | 0.5 | 1 | 0.0 |
| likely | 0.8 | 4.0 | +1.386 |
| certain | 1.0 | $\infty$ | $+\infty$ |

2. Why log-odds, not $p$, as the linear target: $z=w^Tx+b$ is unconstrained in $(-\infty,\infty)$; setting $p=z$ directly breaks the $[0,1]$ bound. Setting $\ln(p/(1-p))=z$ keeps domains matched. $w_j$ = change in log-odds per unit $x_j$; $e^{w_j}$ = odds ratio.
3. Sigmoid, derived from log-odds: $p/(1-p)=e^z \Rightarrow p=e^z(1-p) \Rightarrow p+pe^z=e^z \Rightarrow p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}$. Squashes any real $z$ into $(0,1)$: at $z=0$, $p=0.5$; as $z\to\pm\infty$, $p\to1$ or $0$. Symmetry $\sigma(-z)=1-\sigma(z)$; derivative $\sigma'(z)=\sigma(z)(1-\sigma(z))=p(1-p)$.
4. MLE, single point: $P(y_i|x_i)=p_i^{y_i}(1-p_i)^{1-y_i}$, collapses to $p_i$ when $y_i=1$, $(1-p_i)$ when $y_i=0$. Joint likelihood (i.i.d.): $L(w)=\prod p_i^{y_i}(1-p_i)^{1-y_i}$. Probability fixes $w$, varies $y$ ("how likely are these outcomes"); likelihood fixes $(x,y)$, varies $w$ ("how plausible are these weights").
5. Log-loss, from MLE: log turns the product into a sum, avoids underflow: $\ell(w)=\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$. Gradient descent minimizes, MLE maximizes, negate and average: $L(w)=-\frac{1}{N}\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$.
6. Gradient, chain rule: $\frac{\partial L_i}{\partial w_j}=\frac{\partial L_i}{\partial p_i}\cdot\frac{\partial p_i}{\partial z_i}\cdot\frac{\partial z_i}{\partial w_j}=\left[\frac{p_i-y_i}{p_i(1-p_i)}\right]\cdot[p_i(1-p_i)]\cdot x_{ij}=(p_i-y_i)x_{ij}$. The $p(1-p)$ terms cancel.
7. Batch gradient: $\nabla L=\frac{1}{N}X^T(p-y)$. Update: $w\leftarrow w-\eta\nabla L$. $p>y$ (overestimating) $\Rightarrow$ gradient positive $\Rightarrow$ $w$ decreases; $p<y$ (underestimating) $\Rightarrow$ gradient negative $\Rightarrow$ $w$ increases.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 200
X0 = rng.normal(loc=[-2, -2], scale=1.0, size=(n, 2))
X1 = rng.normal(loc=[2, 2], scale=1.0, size=(n, 2))
X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n), np.ones(n)])

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.6)
plt.title("toy data")
plt.show()

# squashes any real z into (0,1); derivative = p(1-p) -> steep at 0.5, flat at extremes
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# negative log-likelihood of a Bernoulli label (from MLE) -- punishes confident-and-wrong hard
def log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)  # guards log() against log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


In [ ]:
def train_logistic_regression(X, y, lr=0.1, n_iters=1000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)   # start with no opinion, all weights zero
    b = 0.0
    losses = []

    for i in range(n_iters):
        z = X @ w + b               # the linear part: w·x + b, for every row at once
        p = sigmoid(z)               # squash into probabilities

        loss = log_loss(y, p)
        losses.append(loss)

        # (p-y)*x is the derived per-point gradient; dividing by n_samples averages
        # instead of sums, so the update size doesn't depend on how many rows we have
        dw = (X.T @ (p - y)) / n_samples
        db = np.mean(p - y)          # same gradient, but for the bias (x is implicitly 1 here)

        w -= lr * dw                 # step opposite the gradient, that's what makes loss go down
        b -= lr * db

    return w, b, losses

w, b, losses = train_logistic_regression(X, y, lr=0.1, n_iters=1000)

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("training loss over time")
plt.show()

print("learned w:", w)
print("learned b:", b)


In [ ]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression()
sklearn_model.fit(X, y)

print("sklearn w:", sklearn_model.coef_[0])
print("sklearn b:", sklearn_model.intercept_[0])

# ours:    w=[1.936, 1.977], b=0.323
# sklearn: w=[1.884, 1.887], b=0.489
# same sign, same rough ratio between the two features -> confirms our from-scratch
# gradient derivation is correct. Not identical because sklearn adds L2 regularization
# by default and uses a different optimizer, not plain gradient descent.


## Likely Questions

1. Why sigmoid, not some other squashing function? It's the inverse of the logit, the natural link between "linear in log-odds" and "output as a probability." Pairs with log-loss for a clean single-term gradient, $(p-y)x$; a different squashing function wouldn't cancel that cleanly.
2. Why log-loss instead of MSE? Falls directly out of MLE for a Bernoulli label, not chosen by convention. MSE with a sigmoid output is non-convex (gradient descent can get stuck); log-loss with sigmoid is convex, one global minimum guaranteed.
3. How do you interpret a coefficient? Change in log-odds of the positive class per unit increase in that feature, holding others fixed. $e^w$ gives an odds ratio, e.g. $e^w=1.5$ means a 50% increase in the odds of fraud per unit.
4. Why can't logistic regression capture "amount high AND category risky" the way a rule can? $z$ is a plain sum of independently-weighted features, no term for two features acting together unless engineered explicitly. A hand-written AND rule beat logistic regression at equal recall on the fraud project.
5. Why does logistic regression need feature scaling but trees don't? Coefficients are tied to feature magnitude, an unscaled large-range feature dominates the loss from scale alone. Trees split on "is this value above X," scale-invariant.
6. How do you handle severe class imbalance? Weight each sample's loss inversely to its class frequency (`class_weight="balanced"`), or resample. Don't trust a default 0.5 threshold, pick the operating threshold from the actual cost of each error type.
7. Is logistic regression linear? Yes where it matters, the decision boundary ($p=0.5$) is a linear hyperplane, linear in log-odds. Sigmoid itself is nonlinear, but that's just the link function.
8. L1 vs L2 regularization? Both shrink coefficients toward zero. L1 (Lasso) can push some exactly to zero, automatic feature selection. L2 (Ridge, sklearn default) shrinks smoothly, rarely to exactly zero.
